# Notebook 8: Per-Question UMAP Visualizations

Produces a three-panel UMAP figure set for every positive-control question in
`input/Revised_DownSyndrome_60_Questions_Final_linked.xlsx`, using the saved model
answers from notebook 3 (four model arms) and notebooks 5/6 (MedRAG, PaperQA2).

**API:** `http://localhost:8698`  |  **Collection:** `DS_INCLUDE_2026_09_02_DEEPINFRA_2048` (65,677 vectors, 1,594 articles)  |  **Embedding:** `nvidia/llama-nemotron-embed-vl-1b-v2` (DeepInfra, 2048-dim, symmetric encoding)

**Collection guard:** the config cell asserts the collection name, vector count and document
count against both Milvus and the API `/health` endpoint, and stops on any mismatch.

**Run order (no live LLM generation):**
1. **Cell 1**: Imports & Config (always run first)
2. **Shortcut Cell A**: loads `output/ScientificReportRevisionV1/batch_eval/responses_latest.csv`, embeds all texts via the production `/embed` endpoint
3. **Shortcut Cell B**: fits UMAP on the full background corpus, generates the per-question figures

**Output** → `output/ScientificReportRevisionV1/umap/`
- Positive-control questions only; filenames use a `+` prefix and each question's actual SLNo (e.g. `umap_sc_+Q01`), not a sequential renumbering.
- **Six images per question**: `_A` (corpus manifold by section), `_A_legend`, `_B` (retrieved chunks), `_B_legend`, `_C` (model responses vs Ground Truth), `_C_legend`. Legends are separate images, never drawn inside the panels.
- Each image saved as `.pdf` (vector, `rasterized=False` on all scatters) + `.jpg` + `.tiff`, all at 600 dpi.

**Environment:** `dschat` conda environment. Select this kernel before running.


## Cell 1: Imports & Config

In [1]:
import os
import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
import umap
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from ollama import Client as OllamaClient
from pymilvus import connections, Collection
from adjustText import adjust_text

# Load .env from project root
load_dotenv(Path('..') / '.env')

# --- Config ---
EXPECTED_COLLECTION = 'DS_INCLUDE_2026_09_02_DEEPINFRA_2048'   # articles only, stored==searchable since 2026-09-06 purge
EXPECTED_VECTORS    = 65_677
EXPECTED_DOCS       = 1_594
COLLECTION_NAME  = os.getenv('MILVUS_COLLECTION', EXPECTED_COLLECTION)
assert COLLECTION_NAME == EXPECTED_COLLECTION, (
    f'MILVUS_COLLECTION resolved to {COLLECTION_NAME!r}, expected {EXPECTED_COLLECTION!r} - '
    'srv/.env has drifted; update EXPECTED_COLLECTION here if intentional.'
)
MILVUS_URI       = os.getenv('MILVUS_URI', 'http://localhost:19530')
MILVUS_DIM       = int(os.getenv('MILVUS_DIM', '2048'))
T21_API_URL      = os.getenv('T21_API_URL', 'http://localhost:8698/answer')
T21_EMBED_URL    = os.getenv('T21_EMBED_URL', 'http://localhost:8698/embed')

# Nemotron embedding config (used only to describe the model; all embedding done via API)
DEEPINFRA_EMBED_MODEL = os.getenv('DEEPINFRA_EMBED_MODEL', 'nvidia/llama-nemotron-embed-vl-1b-v2')
# QUERY_INSTRUCTION = ''  # symmetric encoding — no instruction prefix for DeepInfra model

# Direct model comparison — OpenAI / Anthropic / Nemotron via Ollama
OPENAI_MODEL    = os.getenv('OPENAI_CHAT_MODEL',    'gpt-5.4')
ANTHROPIC_MODEL = os.getenv('ANTHROPIC_CHAT_MODEL', 'claude-sonnet-4-6')
OLLAMA_API_KEY        = os.getenv('OLLAMA_API_KEY', '')
OLLAMA_NEMOTRON_MODEL = os.getenv('OLLAMA_NEMOTRON_MODEL', 'nemotron-3-super')
OLLAMA_BASE_URL       = os.getenv('OLLAMA_REMOTE_BASE_URL', 'https://ollama.com')

MAX_WORDS  = 150
TOP_K      = 20   # documents to retrieve per query
UMAP_SEED  = 42   # fixed seed for reproducible 2D layout

# Scientific Reports revision (v1) — kept separate from the original submission's
# output/umap/ results. RESPONSES_CSV chains from notebook 3's revised output.
OUTPUT_DIR    = Path('../output/ScientificReportRevisionV1/umap')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESPONSES_CSV = Path('../output/ScientificReportRevisionV1/batch_eval/responses_latest.csv')

OPENAI_KEY    = os.getenv('OPENAI_API_KEY', '')
ANTHROPIC_KEY = os.getenv('ANTHROPIC_API_KEY', '')
assert OPENAI_KEY,    'OPENAI_API_KEY not set in .env'
assert ANTHROPIC_KEY, 'ANTHROPIC_API_KEY not set in .env'
assert OLLAMA_API_KEY, 'OLLAMA_API_KEY not set in .env'

client           = OpenAI(api_key=OPENAI_KEY)
anthropic_client = Anthropic(api_key=ANTHROPIC_KEY)
ollama_client    = OllamaClient(host=OLLAMA_BASE_URL, headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'})

# Connect to Milvus
connections.connect(uri=MILVUS_URI)
collection = Collection(COLLECTION_NAME)
collection.load()

# Three-way consistency check: collection name + vector count + document count
# must match the live API (/health shows the same numbers as the UI sidebar).
_h = requests.get(T21_EMBED_URL.rsplit('/', 1)[0] + '/health', timeout=30).json()
_hv = int(str(_h['database']['total_vectors']).replace(',', ''))
_hd = int(str(_h['database']['estimated_documents']).replace(',', ''))
assert _h['config']['milvus_collection'] == EXPECTED_COLLECTION, (
    f"API serves {_h['config']['milvus_collection']!r}, notebook expects {EXPECTED_COLLECTION!r}")
assert _hv == EXPECTED_VECTORS, f'API vectors {_hv:,} != expected {EXPECTED_VECTORS:,}'
assert _hd == EXPECTED_DOCS,    f'API documents {_hd:,} != expected {EXPECTED_DOCS:,}'
_n_local = collection.query(expr='pk >= 0', output_fields=['count(*)'])[0]['count(*)']
assert _n_local == EXPECTED_VECTORS, f'Milvus count {_n_local:,} != expected {EXPECTED_VECTORS:,}'
print(f'Milvus collection : {COLLECTION_NAME} (verified vs /health)')
print(f'  Vectors         : {_n_local:,} (= API {_hv:,})')
print(f'  Documents       : {_hd:,} (API, articles only)')
print(f'  Vector dim      : {MILVUS_DIM}')
print(f'API endpoint      : {T21_API_URL}')
print(f'Embed endpoint    : {T21_EMBED_URL}  (llama-nemotron-embed-vl-1b-v2 2048-dim, DeepInfra)')
print(f'OpenAI model      : {OPENAI_MODEL}')
print(f'Anthropic model   : {ANTHROPIC_MODEL}')
print(f'Nemotron chat     : {OLLAMA_NEMOTRON_MODEL}  (Ollama @ {OLLAMA_BASE_URL})')
print(f'Responses CSV     : {RESPONSES_CSV}  (exists={RESPONSES_CSV.exists()})')
print(f'Output dir        : {OUTPUT_DIR}')
print('Config OK')


/home/user/anaconda3/envs/dschat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Milvus collection : DS_INCLUDE_2026_09_02_DEEPINFRA_2048 (verified vs /health)
  Vectors         : 65,677 (= API 65,677)
  Documents       : 1,594 (API, articles only)
  Vector dim      : 2048
API endpoint      : http://localhost:8698/answer
Embed endpoint    : http://localhost:8698/embed  (llama-nemotron-embed-vl-1b-v2 2048-dim, DeepInfra)
OpenAI model      : gpt-5.4
Anthropic model   : claude-sonnet-4-6
Nemotron chat     : nemotron-3-super  (Ollama @ https://ollama.com)
Responses CSV     : ../output/ScientificReportRevisionV1/batch_eval/responses_latest.csv  (exists=True)
Output dir        : ../output/ScientificReportRevisionV1/umap
Config OK


## Shortcut Cell B: Per-question three-panel UMAP figures

**Three panels per question**, each with its legend saved as a separate image
(`_A_legend`, `_B_legend`, `_C_legend`), PDF + JPEG + TIFF at 600 dpi:

- **Panel A** (`_A`): background corpus manifold colored by section type: Results, Discussion, Methods, Conclusion, Other. Okabe-Ito colour-blind-safe palette; no red/green pairing.
- **Panel B** (`_B`): background in gray + top-K retrieved chunks colored navy→white by rerank score (horizontal colorbar beneath the plot) + query (red star). The top-3 chunks are annotated with serial letters A/B/C (no periods) in rounded boxes; the separate topic legend uses the identical boxed letter style.
- **Panel C** (`_C`): background in gray + query (red star), Ground Truth (square) and all six model arms (T21 RAG, GPT, Claude, Nemotron, MedRAG, PaperQA2), each with a distinct Okabe-Ito color **and** a distinct marker shape; cosine similarity to Ground Truth is shown in the separate legend. A **magnified inset** shows the neighbourhood where the model responses, Ground Truth and query cluster:
  - the inset is a heavy-edged box (all four sides drawn) kept fully inside the axes, placed in the bottom-right corner by preference, moving to the next corner (top-right, bottom-left, top-left) whenever the magnified region would overlap the box;
  - the magnified (source) region is kept tight so overlapped markers separate clearly in the inset, with only a small floor so the source rectangle stays slightly larger than the query star;
  - the source rectangle and connector lines are drawn in solid black on top of the markers, so the indicator is always visible.

**Output:** `output/ScientificReportRevisionV1/umap/umap_sc_+Qnn_{A,A_legend,B,B_legend,C,C_legend}.{pdf,jpg,tiff}`: six images × three formats per question; axes use 1.8 pt lines with 16 pt bold axis labels and 13 pt tick labels.


## Standalone Shortcut: load saved answers, skip live generation

Run **only Cell 1** (config) first, then the two cells below in order.
No LLM generation and no Milvus retrieval during Cell A; the background corpus is fetched in Cell B.

**Shortcut Cell A**: loads `output/ScientificReportRevisionV1/batch_eval/responses_latest.csv` (positive-control rows, produced by notebook 3) plus the MedRAG and PaperQA2 answers (notebooks 5/6), embeds query + retrieved chunks + all six model responses + Ground Truth via the production `/embed` endpoint, and computes each model's cosine similarity to the Ground Truth per question.

**Shortcut Cell B**: fetches all corpus vectors from Milvus, fits UMAP on the full background corpus, transforms the per-question embeddings into 2D, and saves the six images per question described above to `output/ScientificReportRevisionV1/umap/`.


In [2]:
# ── Shortcut Cell A: Load CSV + Embed all texts via Nemotron API ──────────────
# Prereq: Cell 3 (config/imports) must be run first.
# NO live LLM generation. All text comes from responses_latest.csv.
# Embeds: query, t21_retrieve_1..6, T21/GPT/Claude/Nemotron/MedRAG/PaperQA2 response, GT
# Embedding model: nvidia/llama-nemotron-embed-vl-1b-v2 (2048-dim, DeepInfra) via T21_EMBED_URL

import re   as _sc_re
import json as _sc_json

# ── Text-cleaning helpers ─────────────────────────────────────────────────────

# Matches full citation tokens including INCLUDE-tagged and hyperlinked forms:
#   [PMC12345:Results](url)
#   [PMC12345:Results, INCLUDE](url)
#   [[PMC12345:Results]]
# [^\]]* captures everything inside the brackets (section label, INCLUDE tag, etc.)
_SC_PMC_RE = _sc_re.compile(
    r'\s*\[{1,2}PMC\d{4,}[^\]]*\]{1,2}(?:\([^)]*\))?'
)
_SC_HDR_RE = _sc_re.compile(
    r'^\s*#{1,3}\s*(?:Summary|Key Evidence|Reasoning)[^\n]*\n?',
    flags=_sc_re.M | _sc_re.I,
)

def _sc_clean(txt) -> str:
    if not txt: return ''
    s = str(txt)
    if s.lower() in ('nan', 'none', ''): return ''
    s = _SC_PMC_RE.sub('', s)
    s = _SC_HDR_RE.sub('', s)
    return s.strip()

def _sc_extract_summary(md) -> str:
    """Extract # Summary prose from T21 RAG markdown response."""
    if not md or str(md).lower() in ('nan', 'none', ''): return ''
    m = _sc_re.search(
        r'#{1,2}\s*Summary[^\n]*\n(.*?)(?=\n#{1,2}\s|\Z)',
        str(md), flags=_sc_re.DOTALL | _sc_re.I,
    )
    return _sc_clean(m.group(1).strip() if m else md)

def _sc_embed_q(text: str) -> np.ndarray:
    """Embed query — symmetric encoding, no instruction prefix (DeepInfra model)."""
    r = requests.post(T21_EMBED_URL, json={'text': text, 'mode': 'query'}, timeout=60)
    r.raise_for_status()
    return np.array(r.json()['embedding'], dtype=np.float32)

def _sc_embed_d(text: str) -> np.ndarray:
    """Embed document/response text (no prefix)."""
    r = requests.post(T21_EMBED_URL,
                      json={'text': text or 'no response', 'mode': 'document'},
                      timeout=60)
    r.raise_for_status()
    return np.array(r.json()['embedding'], dtype=np.float32)

def _sc_cosine(a, b) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 0 else 0.0

# ── Verify embed endpoint ─────────────────────────────────────────────────────
_test = requests.post(T21_EMBED_URL, json={'text': 'test', 'mode': 'query'}, timeout=30)
_test.raise_for_status()
assert _test.json()['dim'] == MILVUS_DIM, \
    f"Embed dim mismatch: API={_test.json()['dim']}, expected {MILVUS_DIM}"
print(f'Embed endpoint OK  →  nvidia/llama-nemotron-embed-vl-1b-v2  dim={MILVUS_DIM}  ✓')

# ── Load CSV (positive-control only) ─────────────────────────────────────────
# RESPONSES_CSV is set in Cell 1 — chains from notebook 3's revised output
# (output/ScientificReportRevisionV1/batch_eval/responses_latest.csv). Run
# notebook 3 first if this file doesn't exist yet.
_CSV = RESPONSES_CSV
if not _CSV.exists():
    raise FileNotFoundError(
        f'{_CSV} not found. Run notebook 3 (3_benchmark_pipeline_vs_ground_truth.ipynb) '
        'first to generate it on the revised 60-question benchmark.'
    )
df_sc = pd.read_csv(_CSV)
df_sc.columns = [c.strip() for c in df_sc.columns]
for _col in ['ground_truth', 't21_answer', 'gpt_answer', 'claude_answer',
             'nemotron_answer', 'ctrl_type', 't21_retrieve_meta']:
    if _col in df_sc.columns:
        df_sc[_col] = df_sc[_col].fillna('')
for _col in ['t21_retrieve_1','t21_retrieve_2','t21_retrieve_3',
             't21_retrieve_4','t21_retrieve_5','t21_retrieve_6']:
    if _col in df_sc.columns:
        df_sc[_col] = df_sc[_col].fillna('')

# ── Six-method extension: MedRAG + PaperQA2 answers (articles-only parity runs) ──
_OUT_ROOT = RESPONSES_CSV.parent.parent.parent          # analysis/output/
_med_df = pd.read_csv(_OUT_ROOT / 'medrag_benchmark' / 'medrag_benchmark_results.csv')
_pqa_df = pd.read_csv(_OUT_ROOT / 'paperqa_benchmark' / 'paperqa_benchmark_results.csv')
_sc_normq = lambda s: _sc_re.sub(r'\s+', ' ', str(s).strip().lower())
_med_by_q = {_sc_normq(r['question']): str(r['medrag_answer'] or '')
             for _, r in _med_df.iterrows()}
_pqa_by_q = {_sc_normq(r['question']):
                 str(r['paperqa_clean'] if pd.notna(r.get('paperqa_clean')) else r['paperqa_answer'])
             for _, r in _pqa_df.iterrows()}
print(f'MedRAG answers loaded  : {len(_med_by_q)} (articles-only corpus T21_2026_09_03)')
print(f'PaperQA2 answers loaded: {len(_pqa_by_q)} (articles-only corpus 2026-09-03)')

# Filter to positive controls only
df_sc = df_sc[df_sc['ctrl_type'].str.strip().str.lower().str.startswith('pos')].reset_index(drop=True)
print(f'Loaded {_CSV.name}  →  {len(df_sc)} positive-control questions')
print(f'Retrieve columns: {[c for c in df_sc.columns if c.startswith("t21_retrieve_") and c != "t21_retrieve_meta"]}')
print()

# ── Embed each question ───────────────────────────────────────────────────────
# Per question we embed:
#   query              (mode=query,    instruction prefix)
#   t21_retrieve_1..6  (mode=document, raw retrieved chunk text)
#   t21_answer         (mode=document, Summary section extracted, citations stripped)
#   gpt_answer         (mode=document)
#   claude_answer      (mode=document)
#   nemotron_answer    (mode=document)
#   ground_truth       (mode=document)
#
# Retrieved-doc metadata (score, title, section, pmcid) parsed from t21_retrieve_meta

_RETRIEVE_COLS = ['t21_retrieve_1','t21_retrieve_2','t21_retrieve_3',
                  't21_retrieve_4','t21_retrieve_5','t21_retrieve_6']

all_noret = []

for _, row in df_sc.iterrows():
    q_num    = int(row['sl_no'])
    query    = str(row['question']).strip()
    ctrl_tag = 'POS'
    gt_text  = _sc_clean(str(row['ground_truth']))
    t21_text = _sc_extract_summary(str(row['t21_answer']))
    gpt_text = _sc_clean(str(row['gpt_answer']))
    cld_text = _sc_clean(str(row['claude_answer']))
    nem_text = _sc_clean(str(row['nemotron_answer']))
    med_text = _sc_clean(_med_by_q.get(_sc_normq(query), ''))
    pqa_text = _sc_clean(_pqa_by_q.get(_sc_normq(query), ''))

    # Parse retrieved-doc metadata
    _meta_raw = str(row.get('t21_retrieve_meta', '') or '')
    try:
        _meta = _sc_json.loads(_meta_raw) if _meta_raw.strip().startswith('[') else []
    except Exception:
        _meta = []

    # Collect non-empty retrieved texts
    retrieve_texts = []
    for _col in _RETRIEVE_COLS:
        _txt = str(row.get(_col, '') or '').strip()
        if _txt and _txt.lower() not in ('nan', 'none'):
            retrieve_texts.append(_txt)

    n_ret = len(retrieve_texts)
    print(f'Q{q_num:02d} — {n_ret} retrieved docs | '
          f'T21={len(t21_text.split())}w GPT={len(gpt_text.split())}w '
          f'Claude={len(cld_text.split())}w Nemotron={len(nem_text.split())}w '
          f'MedRAG={len(med_text.split())}w PaperQA2={len(pqa_text.split())}w '
          f'GT={len(gt_text.split())}w')

    # 1. Embed query
    print(f'  [1/{n_ret+6}] query...')
    q_emb = _sc_embed_q(query)

    # 2. Embed retrieved texts (document mode, no prefix)
    ret_embs = []
    for _ii, _rtxt in enumerate(retrieve_texts):
        print(f'  [{_ii+2}/{n_ret+6}] retrieve_{_ii+1}...')
        ret_embs.append(_sc_embed_d(_rtxt))
    ret_embs = np.array(ret_embs, dtype=np.float32) if ret_embs else np.empty((0, MILVUS_DIM), dtype=np.float32)

    # 3. Embed model responses + GT
    print(f'  [{n_ret+2}/{n_ret+6}] T21...')
    t21_emb = _sc_embed_d(t21_text or 'no response')
    print(f'  [{n_ret+3}/{n_ret+6}] GPT...')
    gpt_emb = _sc_embed_d(gpt_text or 'no response')
    print(f'  [{n_ret+4}/{n_ret+6}] Claude...')
    cld_emb = _sc_embed_d(cld_text or 'no response')
    print(f'  [{n_ret+5}/{n_ret+8}] Nemotron...')
    nem_emb = _sc_embed_d(nem_text or 'no response')
    print(f'  [{n_ret+6}/{n_ret+8}] MedRAG...')
    med_emb = _sc_embed_d(med_text or 'no response')
    print(f'  [{n_ret+7}/{n_ret+8}] PaperQA2...')
    pqa_emb = _sc_embed_d(pqa_text or 'no response')
    print(f'  [{n_ret+8}/{n_ret+8}] GT...')
    gt_emb  = _sc_embed_d(gt_text  or 'no information')

    cs_t21 = _sc_cosine(t21_emb, gt_emb)
    cs_gpt = _sc_cosine(gpt_emb, gt_emb)
    cs_cld = _sc_cosine(cld_emb, gt_emb)
    cs_nem = _sc_cosine(nem_emb, gt_emb)
    cs_med = _sc_cosine(med_emb, gt_emb)
    cs_pqa = _sc_cosine(pqa_emb, gt_emb)
    print(f'  cos(GT): T21={cs_t21:.4f} GPT={cs_gpt:.4f} Claude={cs_cld:.4f} '
          f'Nem={cs_nem:.4f} MedRAG={cs_med:.4f} PaperQA2={cs_pqa:.4f}')

    # Retrieve scores, titles, sections, pmcids from meta (for Panel 2 annotation)
    ret_scores  = [float(_m.get('score',  0.5)) for _m in _meta[:n_ret]]
    ret_titles  = [str(_m.get('title',    ''))   for _m in _meta[:n_ret]]
    ret_secs    = [str(_m.get('section',  ''))   for _m in _meta[:n_ret]]
    ret_pmcids  = [str(_m.get('pmcid',    ''))   for _m in _meta[:n_ret]]

    # Pad meta lists if meta shorter than actual retrieved texts
    while len(ret_scores) < n_ret: ret_scores.append(0.5)
    while len(ret_titles) < n_ret: ret_titles.append('')
    while len(ret_secs)   < n_ret: ret_secs.append('')
    while len(ret_pmcids) < n_ret: ret_pmcids.append('')

    all_noret.append({
        'q_num'         : q_num,
        'query'         : query,
        'ctrl_type'     : ctrl_tag,
        'gt_text'       : gt_text,
        't21_text'      : t21_text,
        'gpt_text'      : gpt_text,
        'claude_text'   : cld_text,
        'nemotron_text' : nem_text,
        'medrag_text'   : med_text,
        'paperqa_text'  : pqa_text,
        'retrieve_texts': retrieve_texts,
        'ret_scores'    : ret_scores,
        'ret_titles'    : ret_titles,
        'ret_secs'      : ret_secs,
        'ret_pmcids'    : ret_pmcids,
        'n_retrieved'   : n_ret,
        'query_emb'     : q_emb,
        'ret_embs'      : ret_embs,
        't21_emb'       : t21_emb,
        'gpt_emb'       : gpt_emb,
        'claude_emb'    : cld_emb,
        'nemotron_emb'  : nem_emb,
        'medrag_emb'    : med_emb,
        'paperqa_emb'   : pqa_emb,
        'gt_emb'        : gt_emb,
        'cs_t21'        : cs_t21,
        'cs_gpt'        : cs_gpt,
        'cs_claude'     : cs_cld,
        'cs_nemotron'   : cs_nem,
        'cs_medrag'     : cs_med,
        'cs_paperqa'    : cs_pqa,
    })

print(f'\n✓  {len(all_noret)} questions embedded.')
print('Run Shortcut Cell B to generate UMAP plots.')


Embed endpoint OK  →  nvidia/llama-nemotron-embed-vl-1b-v2  dim=2048  ✓
MedRAG answers loaded  : 30 (articles-only corpus T21_2026_09_03)
PaperQA2 answers loaded: 30 (articles-only corpus 2026-09-03)
Loaded responses_latest.csv  →  30 positive-control questions
Retrieve columns: ['t21_retrieve_1', 't21_retrieve_2', 't21_retrieve_3', 't21_retrieve_4', 't21_retrieve_5', 't21_retrieve_6']

Q01 — 6 retrieved docs | T21=119w GPT=80w Claude=124w Nemotron=77w MedRAG=531w PaperQA2=253w GT=32w
  [1/12] query...


  [2/12] retrieve_1...


  [3/12] retrieve_2...


  [4/12] retrieve_3...


  [5/12] retrieve_4...


...

  [11/14] Nemotron...


  [12/14] MedRAG...


  [13/14] PaperQA2...


  [14/14] GT...


  cos(GT): T21=0.6354 GPT=0.6541 Claude=0.5937 Nem=0.5999 MedRAG=0.6830 PaperQA2=0.6204

✓  30 questions embedded.
Run Shortcut Cell B to generate UMAP plots.


In [3]:

# ── Shortcut Cell B — 3 SEPARATE figures per question (Nature publication style)
# Prereq: Cell 1 (config) + Shortcut Cell A must be run first.
#
# Figure A — Corpus manifold colored by section
# Figure B — Retrieved chunks (navy→white by rerank score) + A./B./C. annotations
# Figure C — Six model responses vs ground truth (cosine similarity)
#
# Style : colour-blind-safe (Okabe–Ito derived) · Arial font · 600 dpi
# Output: PDF + JPEG + TIFF → output/ScientificReportRevisionV1/umap/
#         umap_sc_±Qnn_A.{pdf,jpg,tiff}
#         umap_sc_±Qnn_B.{pdf,jpg,tiff}
#         umap_sc_±Qnn_C.{pdf,jpg,tiff} + umap_sc_±Qnn_C_legend.{pdf,jpg,tiff}

import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap as _LSC

# ── Nature publication matplotlib rcParams ────────────────────────────────────
mpl.rcParams.update({
    'font.family'          : 'sans-serif',
    'font.sans-serif'      : ['Arial', 'Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size'            : 13,
    'axes.titlesize'       : 13,
    'axes.titleweight'     : 'bold',
    'axes.labelsize'       : 16,
    'axes.labelweight'     : 'bold',
    'xtick.labelsize'      : 13,
    'ytick.labelsize'      : 13,
    'legend.fontsize'      : 10,
    'legend.title_fontsize': 11,
    'legend.framealpha'    : 0.95,
    'legend.edgecolor'     : '#CCCCCC',
    'legend.borderpad'     : 0.6,
    'axes.linewidth'       : 1.8,
    'xtick.major.width'    : 1.8,
    'ytick.major.width'    : 1.8,
    'xtick.major.size'     : 5.0,
    'ytick.major.size'     : 5.0,
    'axes.spines.top'      : False,
    'axes.spines.right'    : False,
    'figure.facecolor'     : 'white',
    'axes.facecolor'       : 'white',
    'grid.color'           : '#E5E5E5',
    'grid.linewidth'       : 0.5,
    'grid.linestyle'       : '--',
    'pdf.fonttype'         : 42,   # embed TrueType in PDF
    'ps.fonttype'          : 42,
    'figure.dpi'           : 150,
    'savefig.dpi'          : 600,
})

_DPI = 600

# ── Shared accent colours (not part of the red/green-avoidance concern below) ─
_NPG = {
    'navy'   : '#3C5488',   # retrieved-chunk annotation styling / GT marker
    'cherry' : '#DC0000',   # Query star
}

# Section colours — Panel A — colour-blind-safe (Okabe–Ito derived).
# Previously Results=crimson(red)/Methods=emerald(green) were shown together in
# this same chart; reviewer feedback flagged red/green pairing as both
# inaccessible to common color vision deficiencies and carrying an unintended
# "red=bad / green=good" connotation. No two colors below are a red/green pair.
BG_SECTION_COLORS_SC = {
    'Results'    : ('#E69F00', 'Results',    0.45),   # orange
    'Discussion' : ('#56B4E9', 'Discussion', 0.40),   # sky blue
    'Methods'    : ('#0072B2', 'Methods',    0.45),   # deep blue
    'Conclusion' : ('#CC79A7', 'Conclusion', 0.40),   # reddish-purple
    'Other'      : ('#AAAAAA', 'Other',      0.20),
}

def _scb_sg(section: str) -> str:
    s = str(section or '').strip().lower()
    if s in ('results', 'result'):                                          return 'Results'
    if s in ('discussion', 'discussions'):                                  return 'Discussion'
    if s in ('methods', 'method', 'methodology', 'materials and methods'):  return 'Methods'
    if s in ('conclusion', 'conclusions', 'concluding remarks'):            return 'Conclusion'
    return 'Other'

# Model / point colours + marker shapes — Panel C.
# Reviewer 2 (Figure 5e) flagged Claude (blue diamond) vs T21 (teal diamond) as
# too visually similar to distinguish. Fix: every model now gets BOTH a distinct,
# colour-blind-safe hue (Okabe–Ito) AND a distinct marker shape, so no two models
# can be confused even in grayscale print or by a color-vision-deficient reader.
# Tuple = (fill color, edge color, marker shape)
_OI = {
    'T21'      : ('#0072B2', '#00456B', 'D'),   # blue diamond
    'GPT'      : ('#E69F00', '#8A5D00', '^'),   # orange triangle-up
    'Claude'   : ('#CC79A7', '#7A4863', 'o'),   # reddish-purple circle
    'Nemotron' : ('#D55E00', '#8A3D00', 'v'),   # vermillion triangle-down
    'MedRAG'   : ('#009E73', '#00543E', 'P'),   # bluish-green plus (notebook 20 palette)
    'PaperQA2' : ('#56B4E9', '#2B5F80', 'X'),   # sky-blue cross (notebook 20 palette)
    'GT'       : ('#3C5488', '#1E2A5E', 's'),   # dark navy square
    'Query'    : ('#DC0000', '#8B0000', '*'),   # red star
}

# Navy-to-white ramp for retrieved chunk rerank scores
_NAVY_WHITE = _LSC.from_list(
    'navy_white', [_NPG['navy'], '#3B6BB5', '#90B8E0', '#FFFFFF']
)

_GPT_LBL = OPENAI_MODEL if OPENAI_MODEL else 'GPT'

_ANN_OFFS = [
    ( 55,  35),  # A — upper-right
    (-55,  35),  # B — upper-left
    ( 55, -35),  # C — lower-right
    (-55, -35),  # D — lower-left
    ( 60,   0),  # E — right
    (-60,   0),  # F — left
]


def _save_fig(fig, stem: str) -> None:
    """Save figure as PDF, JPEG, and TIFF at 600 dpi."""
    for fmt, ext in [('pdf', 'pdf'), ('jpeg', 'jpg'), ('tiff', 'tiff')]:
        fig.savefig(
            OUTPUT_DIR / f'{stem}.{ext}',
            dpi=_DPI, bbox_inches='tight', format=fmt, facecolor='white',
        )


# ── Step 1: Fetch all PKs + section labels ────────────────────────────────────
_PAGE  = 16000
_BATCH = 100

print('Step 1/4 — Fetching all PKs + section labels from Milvus...')
_pk_sec, _last_pk = [], -1
while True:
    _page = collection.query(
        expr=f'pk > {_last_pk}', output_fields=['pk', 'section'], limit=_PAGE,
    )
    if not _page: break
    _pk_sec.extend(_page)
    _last_pk = max(r['pk'] for r in _page)
    if len(_page) < _PAGE: break

_all_pks  = [r['pk']             for r in _pk_sec]
_all_secs = [r.get('section','') for r in _pk_sec]
_all_grps = [_scb_sg(s)          for s in _all_secs]
print(f'  Total PKs: {len(_all_pks):,}  |  {dict(Counter(_all_grps))}')

# ── Step 2: Fetch vectors ─────────────────────────────────────────────────────
print(f'\nStep 2/4 — Fetching vectors in batches of {_BATCH}...')
_pk2v: dict = {}
for _i in range(0, len(_all_pks), _BATCH):
    _batch = _all_pks[_i:_i + _BATCH]
    for _r in collection.query(expr=f'pk in {_batch}', output_fields=['pk', 'vector']):
        _pk2v[_r['pk']] = np.array(_r['vector'], dtype=np.float32)
    if (_i // _BATCH) % 100 == 0 and _i > 0:
        print(f'  {min(_i+_BATCH, len(_all_pks)):,}/{len(_all_pks):,}')

_valid   = [pk in _pk2v for pk in _all_pks]
_bg_embs = np.array([_pk2v[pk] for pk, ok in zip(_all_pks, _valid) if ok], dtype=np.float32)
_bg_grps = np.array([g for g, ok in zip(_all_grps, _valid) if ok])
print(f'  Done — {len(_bg_embs):,} vectors  [{_bg_embs.shape[1]}-dim]')

# ── Step 3: Fit UMAP ──────────────────────────────────────────────────────────
print('\nStep 3/4 — Fitting UMAP on background corpus...')
_bg_reducer = umap.UMAP(
    n_neighbors=15, min_dist=0.1, n_components=2,
    metric='cosine', random_state=UMAP_SEED,
)
_bg_2d = _bg_reducer.fit_transform(_bg_embs)
print(f'  UMAP fitted — bg shape: {_bg_2d.shape}')

# ── Step 4: Topic labels from Milvus ─────────────────────────────────────────
print('\nStep 4/4 — Fetching topic labels...')
_all_pmcids = list({
    pid
    for _r in all_noret
    for pid in _r.get('ret_pmcids', [])
    if pid
})
_pmcid_topic: dict = {}
if _all_pmcids:
    _expr = 'pmcid in [' + ', '.join(f'"{p}"' for p in _all_pmcids) + ']'
    for _r in collection.query(
        expr=_expr, output_fields=['pmcid', 'topic', 'mesh_terms'], limit=500,
    ):
        _pid = str(_r.get('pmcid', '') or '').strip()
        if _pid and _pid not in _pmcid_topic:
            _t = str(_r.get('topic', '') or '').strip()
            if not _t:
                _mesh = str(_r.get('mesh_terms', '') or '').strip()
                _t = re.split(r'[;,]', _mesh)[0].strip() if _mesh else ''
            _pmcid_topic[_pid] = _t
print(f'  Resolved {len(_pmcid_topic)}/{len(_all_pmcids)} PMCIDs -> topics')
print()

# ── Per-question: 3 separate Nature-style figures ────────────────────────────
for res in all_noret:
    q_num    = res['q_num']
    ctrl_tag = '-' if res['ctrl_type'] == 'NEG' else '+'
    n_ret    = res['n_retrieved']
    stem     = f'umap_sc_{ctrl_tag}Q{q_num:02d}'
    _q_raw   = res['query']
    q_title  = (f'{ctrl_tag}Q{q_num:02d}: "{_q_raw[:90]}..."'
                if len(_q_raw) > 90 else f'{ctrl_tag}Q{q_num:02d}: "{_q_raw}"')

    _per_q = np.vstack([
        res['query_emb'].reshape(1, -1),
        res['ret_embs'],
        res['t21_emb'].reshape(1, -1),
        res['gpt_emb'].reshape(1, -1),
        res['claude_emb'].reshape(1, -1),
        res['nemotron_emb'].reshape(1, -1),
        res['medrag_emb'].reshape(1, -1),
        res['paperqa_emb'].reshape(1, -1),
        res['gt_emb'].reshape(1, -1),
    ])
    _per_q_2d = _bg_reducer.transform(_per_q)
    _umap_all = np.vstack([_per_q_2d, _bg_2d])

    _q_idx   = 0
    _ret_s   = 1
    _ret_e   = 1 + n_ret
    _t21_idx = _ret_e
    _gpt_idx = _ret_e + 1
    _cld_idx = _ret_e + 2
    _nem_idx = _ret_e + 3
    _med_idx = _ret_e + 4
    _pqa_idx = _ret_e + 5
    _gt_idx  = _ret_e + 6
    _bg_s    = _ret_e + 7
    _bg_xy   = _umap_all[_bg_s:]

    cs_t21 = res['cs_t21']
    cs_gpt = res['cs_gpt']
    cs_cld = res['cs_claude']
    cs_nem = res['cs_nemotron']
    cs_med = res['cs_medrag']
    cs_pqa = res['cs_paperqa']

    # ── Figure A: Corpus manifold by section ─────────────────────────────────
    fig_a, ax_a = plt.subplots(figsize=(6.5, 5.5))
    for _grp, (_col, _lbl, _alp) in BG_SECTION_COLORS_SC.items():
        _msk = _bg_grps == _grp
        if not _msk.any(): continue
        ax_a.scatter(
            _bg_xy[_msk, 0], _bg_xy[_msk, 1],
            c=_col, s=5, alpha=min(_alp + 0.15, 0.85),
            edgecolors='none', zorder=2, rasterized=False,
            label=f'{_lbl}  (n = {_msk.sum():,})',
        )
    ax_a.set_xlabel('UMAP 1', labelpad=4)
    ax_a.set_ylabel('UMAP 2', labelpad=4)
    _handles_a, _labels_a = ax_a.get_legend_handles_labels()
    plt.tight_layout()
    _save_fig(fig_a, f'{stem}_A')
    plt.close(fig_a)
    fig_al = plt.figure(figsize=(3.8, 2.6))
    fig_al.legend(_handles_a, _labels_a, loc='center', ncol=1,
                  markerscale=2.2, title='Section', title_fontsize=11,
                  framealpha=0.95, edgecolor='#BBBBBB')
    _save_fig(fig_al, f'{stem}_A_legend')
    plt.close(fig_al)

    # ── Figure B: Retrieved chunks (navy->white) + Top-3 legend ──────────────
    fig_b, ax_b = plt.subplots(figsize=(6.5, 6.5))
    ax_b.scatter(
        _bg_xy[:, 0], _bg_xy[:, 1],
        c='#888888', s=4, alpha=0.30, edgecolors='none', zorder=1,
        label=f'Background corpus ({len(_bg_grps):,})', rasterized=False,
    )
    if n_ret > 0:
        _scores_raw  = np.array(res['ret_scores'], dtype=np.float32)
        _s_range     = (float(_scores_raw.max()) - float(_scores_raw.min())) or 1.0
        _scores_norm = (_scores_raw - float(_scores_raw.min())) / _s_range
        _sc2 = ax_b.scatter(
            _umap_all[_ret_s:_ret_e, 0], _umap_all[_ret_s:_ret_e, 1],
            c=_scores_norm, cmap=_NAVY_WHITE, vmin=0.0, vmax=1.0,
            s=80, alpha=0.93, edgecolors='#222222', linewidth=0.6, zorder=5,
        )
        _cbar = plt.colorbar(_sc2, ax=ax_b, orientation='horizontal',
                             location='bottom', fraction=0.045, pad=0.12,
                             shrink=0.70)
        _cbar.set_label('Rerank score (normalised)', labelpad=6, fontsize=11)
        _cbar.ax.tick_params(labelsize=10)
        _cbar.outline.set_linewidth(1.2)

        _ann_topics = []
        for _ai in range(min(3, n_ret)):
            _letter = chr(65 + _ai)
            _pmcid  = (res['ret_pmcids'][_ai]
                       if _ai < len(res.get('ret_pmcids', [])) else '')
            _topic  = (_pmcid_topic.get(_pmcid, '').strip()
                       or res['ret_titles'][_ai]
                       or f'chunk #{_ai+1}')
            _ann_topics.append(_topic)
            _ddx, _ddy = _ANN_OFFS[_ai % len(_ANN_OFFS)]
            ax_b.annotate(
                _letter,
                xy=(_umap_all[_ret_s + _ai, 0], _umap_all[_ret_s + _ai, 1]),
                xytext=(_ddx, _ddy), textcoords='offset points',
                fontsize=10, fontweight='bold', color='#111111',
                ha='center', va='center', zorder=20,
                bbox=dict(boxstyle='round,pad=0.30', fc='white',
                          ec=_NPG['navy'], linewidth=0.9, alpha=0.95),
                arrowprops=dict(arrowstyle='->', color=_NPG['navy'],
                                lw=0.9, shrinkA=4, shrinkB=4),
            )

    ax_b.scatter(
        _umap_all[_q_idx, 0], _umap_all[_q_idx, 1],
        c=_OI['Query'][0], s=320, marker=_OI['Query'][2],
        edgecolors=_OI['Query'][1], linewidth=1.2, zorder=30,
        label='Query',
    )
    ax_b.set_xlabel('UMAP 1', labelpad=4)
    ax_b.set_ylabel('UMAP 2', labelpad=4)
    _handles_b, _labels_b = ax_b.get_legend_handles_labels()
    plt.tight_layout()
    _save_fig(fig_b, f'{stem}_B')
    plt.close(fig_b)
    # Legend with the A./B./C. serial letters drawn INSIDE boxes, matching the
    # on-plot annotation style (rounded white box, navy edge).
    from matplotlib.legend_handler import HandlerBase as _HandlerBase
    import matplotlib.text as _mtext
    class _BoxedLetter:
        def __init__(self, letter): self.letter = letter
    class _BoxedLetterHandler(_HandlerBase):
        def create_artists(self, legend, orig, xdescent, ydescent,
                           width, height, fontsize, trans):
            t = _mtext.Text(width / 2 - xdescent, height / 2 - ydescent,
                            orig.letter, fontsize=fontsize * 0.95,
                            ha='center', va='center', fontweight='bold',
                            color='#111111',
                            bbox=dict(boxstyle='round,pad=0.30', fc='white',
                                      ec=_NPG['navy'], linewidth=0.9))
            t.set_transform(trans)
            return [t]
    _bl_handles = list(_handles_b)
    _bl_labels  = list(_labels_b)
    for _ai, _ft in enumerate(_ann_topics):
        _bl_handles.append(_BoxedLetter(f'{chr(65 + _ai)}'))
        _bl_labels.append((_ft[:52] + '...') if len(_ft) > 52 else _ft)
    fig_bl = plt.figure(figsize=(6.4, 2.6))
    fig_bl.legend(_bl_handles, _bl_labels, loc='center', ncol=1,
                  title='Top-3 retrieved chunks', title_fontsize=11,
                  framealpha=0.95, edgecolor='#BBBBBB',
                  handlelength=1.6, labelspacing=0.9, handletextpad=1.0,
                  handler_map={_BoxedLetter: _BoxedLetterHandler()})
    _save_fig(fig_bl, f'{stem}_B_legend')
    plt.close(fig_bl)

    # ── Figure C: Model responses vs ground truth ─────────────────────────────
    # Each model gets a distinct color AND marker shape (see _OI above) so that
    # e.g. Claude and T21 RAG can never be confused, even overlapping near the
    # query star or the ground-truth square (reviewer feedback, Figure 5e).
    fig_c, ax_c = plt.subplots(figsize=(6.5, 5.5))
    ax_c.scatter(
        _bg_xy[:, 0], _bg_xy[:, 1],
        c='#888888', s=4, alpha=0.30, edgecolors='none', zorder=1,
        label=f'Background corpus ({len(_bg_grps):,})', rasterized=False,
    )
    for _key, _idx, _cs, _lbl in [
        ('T21',      _t21_idx, cs_t21, 'T21 RAG'),
        ('GPT',      _gpt_idx, cs_gpt, _GPT_LBL),
        ('Claude',   _cld_idx, cs_cld, 'Claude'),
        ('Nemotron', _nem_idx, cs_nem, 'Nemotron'),
        ('MedRAG',   _med_idx, cs_med, 'MedRAG'),
        ('PaperQA2', _pqa_idx, cs_pqa, 'PaperQA2'),
    ]:
        ax_c.scatter(
            _umap_all[_idx, 0], _umap_all[_idx, 1],
            c=_OI[_key][0], s=95, marker=_OI[_key][2],
            edgecolors=_OI[_key][1], linewidth=0.8, zorder=10,
            label=f'{_lbl}  ({_OI[_key][2]} marker, cos = {_cs:.3f})',
        )
    ax_c.scatter(
        _umap_all[_gt_idx, 0], _umap_all[_gt_idx, 1],
        c=_OI['GT'][0], s=100, marker=_OI['GT'][2],
        edgecolors=_OI['GT'][1], linewidth=0.8, zorder=10,
        label='Ground Truth  (square marker)',
    )
    ax_c.scatter(
        _umap_all[_q_idx, 0], _umap_all[_q_idx, 1],
        c=_OI['Query'][0], s=320, marker=_OI['Query'][2],
        edgecolors=_OI['Query'][1], linewidth=1.2, zorder=30,
        label=f'Query  (star marker, cos = {_sc_cosine(res["query_emb"], res["gt_emb"]):.3f})',
    )
    # ── Zoom inset: neighbourhood of all annotated points (models, GT, query) ──
    _foc = np.array([_umap_all[_i] for _i in
                     [_t21_idx, _gpt_idx, _cld_idx, _nem_idx, _med_idx, _pqa_idx,
                      _gt_idx, _q_idx]])
    _fx0, _fy0 = _foc.min(axis=0); _fx1, _fy1 = _foc.max(axis=0)
    _zpad = 0.35 * max(_fx1 - _fx0, _fy1 - _fy0, 1e-6)
    _zx0, _zx1 = _fx0 - _zpad, _fx1 + _zpad
    _zy0, _zy1 = _fy0 - _zpad, _fy1 + _zpad
    _fcx, _fcy = _foc.mean(axis=0)
    _x_lo, _x_hi = ax_c.get_xlim(); _y_lo, _y_hi = ax_c.get_ylim()
    # keep the source region TIGHT (strong magnification, overlapped markers
    # clearly separated in the zoom box); only a small floor so the rectangle
    # stays barely larger than the query star
    _zcx, _zcy = (_zx0 + _zx1) / 2, (_zy0 + _zy1) / 2
    _min_half = 0.022 * max(_x_hi - _x_lo, _y_hi - _y_lo)
    _half = max((_zx1 - _zx0) / 2, (_zy1 - _zy0) / 2, _min_half)
    _zx0, _zx1 = _zcx - _half, _zcx + _half
    _zy0, _zy1 = _zcy - _half, _zcy + _half
    # inset prefers the bottom-right corner but must NOT overlap the magnified
    # (source) region: try corners in preference order and take the first whose
    # box is clear of the zoom region; fall back to the least-overlapping one.
    def _frac(_vx, _vy):
        return ((_vx - _x_lo) / (_x_hi - _x_lo), (_vy - _y_lo) / (_y_hi - _y_lo))
    _zfx0, _zfy0 = _frac(_zx0, _zy0)
    _zfx1, _zfy1 = _frac(_zx1, _zy1)
    _m = 0.03   # clearance margin in axes fraction
    def _overlap_area(_bx, _by, _bw=0.39, _bh=0.39):
        _ox = max(0.0, min(_bx + _bw + _m, _zfx1) - max(_bx - _m, _zfx0))
        _oy = max(0.0, min(_by + _bh + _m, _zfy1) - max(_by - _m, _zfy0))
        return _ox * _oy
    _corner_pref = [(0.56, 0.05), (0.56, 0.56), (0.05, 0.05), (0.05, 0.56)]
    _best = min(_corner_pref, key=lambda k: (_overlap_area(*k), _corner_pref.index(k)))
    axins = ax_c.inset_axes([_best[0], _best[1], 0.39, 0.39])
    axins.scatter(_bg_xy[:, 0], _bg_xy[:, 1], c='#888888', s=4, alpha=0.30,
                  edgecolors='none', zorder=1, rasterized=False)
    for _key, _idx in [('T21', _t21_idx), ('GPT', _gpt_idx), ('Claude', _cld_idx),
                       ('Nemotron', _nem_idx), ('MedRAG', _med_idx), ('PaperQA2', _pqa_idx)]:
        axins.scatter(_umap_all[_idx, 0], _umap_all[_idx, 1], c=_OI[_key][0], s=130,
                      marker=_OI[_key][2], edgecolors=_OI[_key][1], linewidth=0.9,
                      zorder=10, rasterized=False)
    axins.scatter(_umap_all[_gt_idx, 0], _umap_all[_gt_idx, 1], c=_OI['GT'][0], s=140,
                  marker=_OI['GT'][2], edgecolors=_OI['GT'][1], linewidth=0.9,
                  zorder=10, rasterized=False)
    axins.scatter(_umap_all[_q_idx, 0], _umap_all[_q_idx, 1], c=_OI['Query'][0], s=340,
                  marker=_OI['Query'][2], edgecolors=_OI['Query'][1], linewidth=1.3,
                  zorder=30, rasterized=False)
    axins.set_xlim(_zx0, _zx1); axins.set_ylim(_zy0, _zy1)
    axins.set_xticks([]); axins.set_yticks([])
    for _sp in axins.spines.values():
        _sp.set_visible(True)   # rcParams hide top/right spines; the inset box needs all four
        _sp.set_edgecolor('#111111'); _sp.set_linewidth(2.2)
    # zoom indicator: solid black source rectangle (all four sides) + black connectors
    _zrect, _zconns = ax_c.indicate_inset_zoom(axins, edgecolor='#000000', linewidth=2.2)
    _zrect.set_edgecolor('#000000'); _zrect.set_linewidth(2.2); _zrect.set_alpha(1.0)
    _zrect.set_zorder(40)   # draw the source rectangle ON TOP of the markers
    for _cl in _zconns:
        _cl.set_color('#000000'); _cl.set_linewidth(1.8); _cl.set_alpha(1.0)
        _cl.set_zorder(39)

    ax_c.set_xlabel('UMAP 1', labelpad=4)
    ax_c.set_ylabel('UMAP 2', labelpad=4)
    # Legend saved as a SEPARATE figure (like the notebook-8 UMAP legends) so the
    # scatter fills the full canvas; compose them side by side in the manuscript.
    _handles_c, _labels_c = ax_c.get_legend_handles_labels()
    plt.tight_layout()
    _save_fig(fig_c, f'{stem}_C')
    plt.close(fig_c)

    fig_cl = plt.figure(figsize=(4.8, 3.0))
    _leg = fig_cl.legend(
        _handles_c, _labels_c, loc='center', ncol=1,
        title='Cosine similarity to GT (marker shape identifies model)',
        title_fontsize=10, fontsize=10,
        framealpha=0.95, edgecolor='#BBBBBB',
    )
    fig_cl.gca().set_axis_off() if fig_cl.axes else None
    _save_fig(fig_cl, f'{stem}_C_legend')
    plt.close(fig_cl)

    print(f'{stem}  A/B/C  ->  PDF+JPEG+TIFF  |  '
          f'T21={cs_t21:.4f}  {_GPT_LBL}={cs_gpt:.4f}  '
          f'Claude={cs_cld:.4f}  Nem={cs_nem:.4f}  '
          f'MedRAG={cs_med:.4f}  PaperQA2={cs_pqa:.4f}')

print(f'\n DONE: {len(all_noret)} x 3 figures x 3 formats = '
      f'{len(all_noret)*9} files saved (600 dpi) -> {OUTPUT_DIR}')


Step 1/4 — Fetching all PKs + section labels from Milvus...


  Total PKs: 65,677  |  {'Methods': 16811, 'Discussion': 16268, 'Results': 19886, 'Other': 11431, 'Conclusion': 1281}

Step 2/4 — Fetching vectors in batches of 100...


  10,100/65,677


  20,100/65,677


  30,100/65,677


...

umap_sc_+Q28  A/B/C  ->  PDF+JPEG+TIFF  |  T21=0.6880  gpt-5.4=0.6634  Claude=0.6919  Nem=0.6481  MedRAG=0.7165  PaperQA2=0.0594


/tmp/ipykernel_689623/1380856855.py:435: MatplotlibDeprecationWarning: Since Matplotlib 3.10 indicate_inset_[zoom] returns a single InsetIndicator artist with a rectangle property and a connectors property.  From 3.12 it will no longer be possible to unpack the return value into two elements.
  _zrect, _zconns = ax_c.indicate_inset_zoom(axins, edgecolor='#000000', linewidth=2.2)


umap_sc_+Q29  A/B/C  ->  PDF+JPEG+TIFF  |  T21=0.6180  gpt-5.4=0.7143  Claude=0.7049  Nem=0.6762  MedRAG=0.6700  PaperQA2=0.7190


/tmp/ipykernel_689623/1380856855.py:435: MatplotlibDeprecationWarning: Since Matplotlib 3.10 indicate_inset_[zoom] returns a single InsetIndicator artist with a rectangle property and a connectors property.  From 3.12 it will no longer be possible to unpack the return value into two elements.
  _zrect, _zconns = ax_c.indicate_inset_zoom(axins, edgecolor='#000000', linewidth=2.2)


umap_sc_+Q30  A/B/C  ->  PDF+JPEG+TIFF  |  T21=0.6354  gpt-5.4=0.6541  Claude=0.5937  Nem=0.5999  MedRAG=0.6830  PaperQA2=0.6204

 DONE: 30 x 3 figures x 3 formats = 270 files saved (600 dpi) -> ../output/ScientificReportRevisionV1/umap
